# 08 — Efficiency roll-over at I > 10²⁰ W/cm²

Reproduces Fig. 2a of Timmis 2026.  In the efficiency-limit regime the
absolute conversion η plateaus rather than following the naive a₀² growth
of the relativistic-spikes cutoff.  We sweep the driver intensity and
measure the integrated η over harmonics 12–47.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from harmonyemissions import Laser, Target, simulate
from harmonyemissions.config import NumericsConfig
from harmonyemissions.units import a0_to_intensity

In [ ]:
numerics = NumericsConfig(pipeline_grid=96, pipeline_dx_um=0.1)
a0s = np.array([2, 5, 10, 20, 40, 80])
etas = []
intensities = [a0_to_intensity(a, 0.8) for a in a0s]
for a0 in a0s:
    r = simulate(Laser(a0=float(a0), spatial_profile='super_gaussian',
                       spot_fwhm_um=2.0, angle_deg=45.0),
                 Target.sio2(t_HDR_fs=351.0),
                 model='surface_pipeline', numerics=numerics)
    n = r.spectrum.coords['harmonic'].values
    mask = (n >= 12) & (n <= 47)
    etas.append(float(r.spectrum.values[mask].sum()))
etas = np.array(etas)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(intensities, etas / etas.max(), 'o-')
ax.axvspan(1e20, max(intensities) * 1.5, alpha=0.15, color='red', label='efficiency-limit regime')
ax.set_xlabel('driver intensity [W/cm²]')
ax.set_ylabel('integrated η_{12–47} (normalised)')
ax.set_title('Efficiency roll-over at I > 10²⁰ W/cm²')
ax.legend()
plt.show()